# Notebook 05 Statistical Inference Laboratory

Notebook 04 treated distributions, likelihood, and maximum likelihood estimation. Those tools produce a *point estimate* $\hat\theta$ from a finite sample. This notebook asks the next question: **how much should we trust an estimate computed from $n$ observations?**

Statistical inference is the machinery that turns a sample into a statement about an unknown population, while keeping the uncertainty of that statement explicit.

Our journey:
$$
\boxed{
\text{Population}
\rightarrow
\text{Sample}
\rightarrow
\text{Estimator}
\rightarrow
\text{Sampling Distribution}
\rightarrow
\text{Standard Error}
\rightarrow
\text{Confidence Interval}
\rightarrow
\text{Hypothesis Test}
\rightarrow
\text{ANOVA}
}
$$

Every major claim below is derived, implemented from the formula, checked against a trusted library, and then shown in a controlled experiment.


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.datasets import fetch_california_housing

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.linear_algebra_and_stat import sampling_utils
from src.linear_algebra_and_stat import estimator_utils
from src.linear_algebra_and_stat import hypothesis_testing
from src.linear_algebra_and_stat import visualization_utils

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 110


## 1. Sampling Distributions and the Standard Error

When we draw a sample, we do not get "the population." We get one random subset. Change the sample and every statistic computed from it changes with it.

That observation is the entire starting point of inference:

* each observation $X_i$ is a random variable
* the sample $(X_1,\dots,X_n)$ is therefore random
* any statistic $T(X_1,\dots,X_n)$ is itself a random variable
* in particular the sample mean $\bar X$ is a random variable, so it has an expectation and a variance

Let $X_1,X_2,\ldots,X_n$ be independent observations from a population with
$$
E[X_i]=\mu, \qquad Var(X_i)=\sigma^2.
$$
Define
$$
\bar X=\frac{1}{n}\sum_{i=1}^{n}X_i.
$$

### Expectation of the sample mean

Linearity of expectation does **not** require independence:
$$
E[\bar X]
=
E\left[\frac{1}{n}\sum_{i=1}^n X_i\right]
=
\frac{1}{n}\sum_{i=1}^n E[X_i]
=
\frac{1}{n}\cdot n\mu
=
\mu.
$$
$$
\boxed{E[\bar X]=\mu}
$$
So $\bar X$ is centered on the population mean. That is already a strong statement: the sampling distribution of the mean is not some new unknown location; it sits at $\mu$.

### Variance of the sample mean

Variance *does* need independence (or at least uncorrelatedness) to drop the cross terms. First use $Var(aX)=a^2 Var(X)$:
$$
Var(\bar X)
=
Var\left(\frac{1}{n}\sum_{i=1}^n X_i\right)
=
\frac{1}{n^2}\,Var\left(\sum_{i=1}^n X_i\right).
$$
Independence gives $Var(\sum_i X_i)=\sum_i Var(X_i)$:
$$
Var(\bar X)
=
\frac{1}{n^2}\sum_{i=1}^n \sigma^2
=
\frac{n\sigma^2}{n^2}
=
\frac{\sigma^2}{n}.
$$
$$
\boxed{Var(\bar X)=\frac{\sigma^2}{n}}
$$
The standard deviation of that sampling distribution is the **standard error of the mean**:
$$
\boxed{SE(\bar X)=\frac{\sigma}{\sqrt n}}
$$

> **Crucial distinction:** Standard deviation $\sigma$ describes the variability of *individual observations*. Standard error $\sigma/\sqrt n$ describes the variability of a *statistic* across repeated samples. They are not interchangeable.

In practice $\sigma$ is unknown, so we replace it by the sample standard deviation $s$ and use
$$
\widehat{SE}(\bar X)=\frac{s}{\sqrt n}.
$$
That substitution is not free: it injects extra uncertainty, which is why Student's $t$ appears later.

### The $1/\sqrt n$ law

$$
SE \propto \frac{1}{\sqrt n}.
$$
To reduce uncertainty by a factor of $x$ we need about $x^2$ times as much data:

* $2\times$ lower uncertainty $\rightarrow$ $4\times$ the observations
* $3\times$ lower uncertainty $\rightarrow$ $9\times$ the observations
* $10\times$ lower uncertainty $\rightarrow$ $100\times$ the observations

Additional data helps, but slowly. That is why "just collect more data" is a real cost, not a slogan.


In [ ]:
# Experiment: SE shrinks like 1/sqrt(n), not 1/n.
visualization_utils.plot_se_shrinkage(sigma=1.0, n_max=400)


The curve falls quickly at first and then flattens. Doubling $n$ from $4$ to $8$ helps a lot; doubling from $200$ to $400$ barely moves the standard error. This is the same $1/\sqrt n$ that will later control the width of every confidence interval and the power of every $t$-test in this notebook.

## 2. Sampling-Distribution Laboratory

A statistic can behave completely differently from the observations it is computed from. The cleanest way to see this is to fix a population, draw many samples of size $n$, compute $\bar X$ each time, and look at the histogram of those means.

We do this for four populations:

1. **Normal** — already bell-shaped
2. **Uniform** — flat, no tails
3. **Exponential** — right-skewed
4. **Log-normal** — heavily skewed

For each population we will:

* look at the original distribution
* draw thousands of samples at $n \in \{5, 30, 100\}$
* plot the sampling distribution of the mean
* overlay the theoretical $N(\mu, \sigma/\sqrt n)$ curve
* compare empirical mean, variance, and SE with the formulas from Section 1

The population itself is **not** becoming normal. Only the histogram of *averages* is.


In [ ]:
# Four synthetic populations, treated as finite empirical worlds.
populations = sampling_utils.make_synthetic_populations(n_pop=100_000, seed=42)

# Small n: irregular. Moderate n: CLT appearing. Large n: tight and nearly Gaussian.
visualization_utils.plot_sampling_distribution_convergence(populations, [5, 30, 100], n_reps=1500, seed=42)
visualization_utils.print_sampling_comparison(populations, sample_size=30, n_reps=2500, seed=42)


**How to read that grid.**

* Column 0 is the *population*. Exponential and log-normal stay skewed no matter how large $n$ is.
* The later columns are histograms of $\bar X$, not of $X$. They become smoother, more symmetric, and narrower.
* The red curve is not a fit we hoped for: it is the exact normal predicted by $E[\bar X]=\mu$ and $SE=\sigma/\sqrt n$.
* The printed table is the Monte Carlo check of Section 1. Empirically, $E[\bar X]\approx\mu$ and $SE(\bar X)\approx\sigma/\sqrt n$ for every population, including the badly skewed ones.

That last sentence is the content of the Central Limit Theorem, which we can now state with experimental intuition already in hand.

## 3. Central Limit Theorem

Under standard conditions (independent observations, finite variance, $n$ not tiny),
$$
\bar X \;\approx\; N\left(\mu,\;\frac{\sigma^2}{n}\right)
$$
or, writing the second parameter as a standard deviation,
$$
\boxed{\bar X \;\approx\; N\left(\mu,\;\frac{\sigma}{\sqrt n}\right)}.
$$

Two effects happen at once as $n$ grows:

1. the sampling distribution becomes *more normal*
2. its standard error *shrinks*

They are different effects. A uniform population is already symmetric, so even $n=5$ looks roughly Gaussian — the visible change is mostly the shrinking width. A log-normal population is still visibly skewed at $n=5$; by $n=100$ the histogram of means is almost indistinguishable from the red normal curve, *and* it is much narrower.

> **Crucial distinction:** The CLT does **not** say that the original data become normal. Individual house prices, waiting times, or neural-net accuracies can stay wild. Averages of those quantities become predictable.

This is why machine learning metrics computed on a test set of size $n$ are much more stable than any single prediction, and why their uncertainty still scales only like $1/\sqrt n$.

## 4. A Real-World Population

Synthetic Gaussians are convenient. They are also a little too polite. We now treat a real dataset as an empirical population, then sample from it exactly as we did above.

**Dataset.** [California Housing](https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html) (Pace & Barry, 1997), shipped by scikit-learn as `fetch_california_housing`. Each of the $20{,}640$ rows is a census *block group*. We use the target `MedHouseVal`: median house value in units of $\$100{,}000$.

**Why this variable.** It is continuous, has a large $N$, has obvious spread, and is right-skewed (a long expensive tail, plus a cap near $5.0$). No feature engineering is required: we are not predicting anything. The full vector of $20{,}640$ values *is* the population for the rest of the experiment.

**Preprocessing.** None, beyond reading the target. We do not drop the cap, because that cap is part of the real sampling distribution.

In [ ]:
california = fetch_california_housing()
housing = california.target  # median house value, units of $100,000

mu_h, var_h, sd_h = sampling_utils.population_moments(housing)
print("Source: Pace & Barry (1997), California Housing via sklearn")
print(f"N = {len(housing)} block groups")
print(f"Population mean μ  = {mu_h:.4f}")
print(f"Population std  σ  = {sd_h:.4f}")
print(f"Skewness           = {stats.skew(housing):.3f}  (right-skewed)")
print(f"Min / max          = {housing.min():.3f} / {housing.max():.3f}")

visualization_utils.plot_real_population_and_samples(
    housing, "California MedHouseVal", sample_sizes=(5, 30, 100), n_reps=1500, seed=1
)
visualization_utils.print_sampling_comparison(
    {"California MedHouseVal": housing}, sample_size=30, n_reps=2500, seed=1
)


The population histogram is skewed and piled up near the $500{,}000$ cap. The histogram of sample means at $n=100$ is already close to Gaussian and is centered on the true mean $2.07$. The formulas did not require the data to be normal; they required a finite variance and independent draws from this empirical world.

We will come back to these house prices when we build confidence intervals.

## 5. Estimators: What Makes One Better Than Another?

An **estimator** is a *rule*, a function of the sample. An **estimate** is the number you get after applying that rule to one particular sample.

$$\hat\theta = t(X_1,\dots,X_n) \qquad\text{is an estimator of a parameter}\qquad \theta. $$

The sample mean $\bar X$ is an estimator of $\mu$. The number $2.14$ computed from one housing sample is an estimate.

How do we compare rules? The first criterion is bias.

### Bias

$$
\boxed{Bias(\hat\theta)=E[\hat\theta]-\theta}
$$

$\theta$ is a theoretical reference. In a real problem we do not know it; in a laboratory we *do*, which is the only reason we can measure bias.

For the sample mean:
$$
E[\hat\mu]=E[\bar X]=\mu \implies Bias(\bar X)=0.
$$
The sample mean is an **unbiased** estimator of the population mean.

> **Unbiased does not mean perfect.** It means that if we could repeat the experiment forever, the average of the estimates would hit $\theta$. A single estimate can still be terrible.

A useful foil is the first-observation estimator $\hat\theta=X_1$. Then $E[X_1]=\mu$, so it is also unbiased, but
$$
Var(X_1)=\sigma^2,
$$
which **does not shrink with $n$**. Meanwhile
$$
Var(\bar X)=\sigma^2/n \to 0.
$$
Unbiasedness without vanishing variance is not enough.

The opposite failure mode is an estimator that is *consistently wrong*: low variance, nonzero bias. A shrinkage rule
$$
\hat\theta_\lambda = \lambda\bar X + (1-\lambda)\,c
$$
with $\lambda<1$ pulls every estimate toward a constant $c$. Its sampling distribution is tighter than $\bar X$, but it is centered in the wrong place unless $c=\mu$.

* **Low bias, high variance:** unstable, scattered around the truth ($X_1$).
* **High bias, low variance:** tightly clustered, systematically off ($\hat\theta_\lambda$).
* **Low bias, low variance:** the sample mean, and the target we want.

### Bessel's correction, and why $n-1$ appears

If we knew $\mu$, $\frac{1}{n}\sum_i (X_i-\mu)^2$ would be unbiased for $\sigma^2$. We do not know $\mu$, so we plug in $\bar X$. Because $\bar X$ is computed from the same sample, it sits in the exact center of that sample and the residuals $(X_i-\bar X)$ are a little too small:
$$
E\left[\frac{1}{n}\sum_i(X_i-\bar X)^2\right]=\frac{n-1}{n}\sigma^2.
$$
Multiplying by $n/(n-1)$ — equivalently, dividing by $n-1$ — removes the bias:
$$
s^2=\frac{1}{n-1}\sum_i(X_i-\bar X)^2.
$$
We have used one degree of freedom to estimate the mean, which is the same $n-1$ that will appear in the $t$ distribution. Notebook 02 already met this convention in the covariance matrix of PCA.

In [ ]:
# Experiment: three estimators of μ on N(10, 5²) samples of size n=20.
rng_seed = 42
n_reps, n, mu, sigma = 2500, 20, 10.0, 5.0
samples = sampling_utils.repeated_samples(
    sampling_utils.sample_normal, n_samples=n_reps, sample_size=n,
    seed=rng_seed, mu=mu, sigma=sigma
)

# The shrunk estimator pulls toward 0, so for μ=10 it is clearly biased.
estimators = {
    r"$X_1$ (unbiased, high var)": estimator_utils.first_observation,
    r"$\bar{X}$ (unbiased)": estimator_utils.sample_mean,
    r"$0.65\bar{X}$ (biased, low var)": lambda x: estimator_utils.shrunk_mean(x, shrinkage=0.65, target=0.0),
}

results = {name: estimator_utils.evaluate_estimator(fn, samples, mu) for name, fn in estimators.items()}
visualization_utils.plot_estimator_comparison(results, true_parameter=mu)

print(f"{'Estimator':<28} {'bias':>8} {'var':>8} {'MSE':>8} {'var+bias²':>10}")
for name, res in results.items():
    print(f"{name:<28} {res['bias']:8.3f} {res['variance']:8.3f} {res['mse']:8.3f} {res['mse_from_identity']:10.3f}")

# Small-n check of Bessel's correction.
true_var = sigma ** 2
tiny = sampling_utils.repeated_samples(
    sampling_utils.sample_normal, n_samples=3000, sample_size=5,
    seed=0, mu=mu, sigma=sigma
)
biased = [estimator_utils.biased_variance(s) for s in tiny]
unbiased = [estimator_utils.unbiased_variance(s) for s in tiny]
print()
print(f"True σ² = {true_var:.1f}")
print(f"Mean of (1/n)Σ(Xi-X̄)²     = {np.mean(biased):.3f}   (theory {(4/5)*true_var:.3f})")
print(f"Mean of (1/(n-1))Σ(Xi-X̄)² = {np.mean(unbiased):.3f}   (theory {true_var:.3f})")


The left panel ($X_1$) is centered on $10$ and very wide: unbiased, uselessly noisy. The middle panel ($\bar X$) is centered on $10$ and much tighter. The right panel ($0.65\bar X$) is tighter still, but its mass sits left of $10$. That is the picture of "consistently close to the *wrong* number."

The $n=5$ variance experiment confirms Bessel's formula: dividing by $n$ systematically underestimates $\sigma^2$; dividing by $n-1$ does not.

## 6. Mean Squared Error

Bias and variance are different failure modes. A single number that combines them is
$$
MSE(\hat\theta)=E[(\hat\theta-\theta)^2].
$$

### Derivation of $MSE = Var + Bias^2$

$\theta$ is a constant. $\hat\theta$ is a random variable. Add and subtract $E[\hat\theta]$:
$$
\hat\theta-\theta
=
\bigl(\hat\theta-E[\hat\theta]\bigr)
+
\bigl(E[\hat\theta]-\theta\bigr).
$$
Square:
$$
(\hat\theta-\theta)^2
=
(\hat\theta-E[\hat\theta])^2
+
(E[\hat\theta]-\theta)^2
+
2(\hat\theta-E[\hat\theta])(E[\hat\theta]-\theta).
$$
Take expectation. The first term is $Var(\hat\theta)$ by definition. The second term is a constant, so
$$
E[(E[\hat\theta]-\theta)^2]=(E[\hat\theta]-\theta)^2=Bias(\hat\theta)^2.
$$
The cross term vanishes because $E[\hat\theta]-\theta$ is constant and $E[\hat\theta-E[\hat\theta]]=0$:
$$
2(E[\hat\theta]-\theta)\,E[\hat\theta-E[\hat\theta]]=0.
$$
$$
\boxed{MSE(\hat\theta)=Var(\hat\theta)+Bias(\hat\theta)^2}
$$

This identity is not a slogan. We now verify it on the Monte Carlo estimates we just computed.

In [ ]:
# Numerical verification of MSE = Var + Bias² on the shrinkage estimator
# (the interesting case: both terms are nonzero).
shrunk_estimates = results[r"$0.65\bar{X}$ (biased, low var)"]["estimates"]
check = estimator_utils.mse_identity_holds(shrunk_estimates, mu)
visualization_utils.plot_mse_identity(check)
print(check)


The bar for $MSE$ matches the bar for $Var+Bias^2$ up to floating-point noise. The identity is doing real arithmetic, not decoration.

### Connection to the ML bias–variance tradeoff

The same decomposition is why we talk about bias and variance of *models*:

* an overly simple model (linear fit to a curved signal, a stump, a tiny network) is **high bias**, systematically missing the target
* an overly flexible model (high-degree polynomial, deep net on tiny data) is **high variance**, it jumps when the training sample jumps
* a useful model balances the two, which is exactly minimizing something like MSE

This notebook will not train a zoo of models. The estimator experiment is the statistical skeleton of that later ML conversation: "centered on the truth" and "close to the truth" are not the same picture.

## 7. Consistency

Unbiasedness is a statement at a *fixed* $n$. **Consistency** is a statement about the limit $n\to\infty$: $$\hat\theta_n \xrightarrow{p} \theta.$$
For the sample mean this is the Law of Large Numbers from Notebook 04: $$\bar X \xrightarrow{p} \mu.$$

These properties are not interchangeable:

| Property | Meaning | Counterexample if you drop the others |
|---|---|---|
| Unbiased | $E[\hat\theta]=\theta$ at finite $n$ | $X_1$ is unbiased and does not concentrate |
| Low variance | estimates cluster | a constant $7$ has zero variance and is biased |
| Small MSE | $Var+Bias^2$ is small | can still be inconsistent if it does not improve with $n$ |
| Consistent | concentrates on $\theta$ as $n\to\infty$ | the biased variance $s_n^2$ (divide by $n$) *is* consistent, even though it is biased |

The first-observation estimator is unbiased for every $n$ and is **not** consistent. The MLE variance $s_n^2$ is biased for every finite $n$ and **is** consistent, because $(n-1)/n\to 1$.

In [ ]:
# Consistency laboratory: X̄ concentrates; X₁ does not.
sample_sizes = [5, 15, 40, 120, 400]
mean_traj = estimator_utils.consistency_trajectories(
    estimator_utils.sample_mean,
    lambda size, seed: sampling_utils.sample_normal(mu=10, sigma=5, size=size, seed=seed),
    true_parameter=10.0, sample_sizes=sample_sizes, n_reps=250, seed=1
)
x1_traj = estimator_utils.consistency_trajectories(
    estimator_utils.first_observation,
    lambda size, seed: sampling_utils.sample_normal(mu=10, sigma=5, size=size, seed=seed),
    true_parameter=10.0, sample_sizes=sample_sizes, n_reps=250, seed=1
)
print("Sample mean — clouds should collapse onto 10")
visualization_utils.plot_consistency(mean_traj, 10.0, ylabel=r"$\bar X$")
print("First observation — clouds stay as wide as the original population")
visualization_utils.plot_consistency(x1_traj, 10.0, ylabel=r"$X_1$")


## 8. Confidence Intervals and Student's $t$

We now have an estimator and a way to score it. We still need a way to *quantify uncertainty around one observed estimate*.

If $X_i\sim N(\mu,\sigma^2)$ and $\sigma$ is known,
$$
Z=\frac{\bar X-\mu}{\sigma/\sqrt n}\sim N(0,1),
$$
and the familiar $95\%$ interval uses $z_{0.975}\approx 1.96$:
$$
\bar X \pm 1.96\cdot\frac{\sigma}{\sqrt n}.
$$
About $95\%$ of a normal's mass sits within $1.96$ standard deviations of its mean. Here the relevant standard deviation is the **standard error**, not $\sigma$.

Where does $1.96$ come from? It is the inverse CDF of $N(0,1)$:
$$
\boxed{z_{1-\alpha/2}=F_{N(0,1)}^{-1}(1-\alpha/2)=\Phi^{-1}(1-\alpha/2).}
$$
For $\alpha=0.05$, $\Phi^{-1}(0.975)\approx 1.960$. In code this is `stats.norm.ppf(0.975)`. The CDF answers "how much mass lies to the left of this $z$"; the PPF (quantile function) answers the inverse question: "which $z$ has $97.5\%$ of the mass to its left?"

### The unknown-$\sigma$ problem

We almost never know $\sigma$. Replacing it by $s$ gives
$$
T=\frac{\bar X-\mu}{s/\sqrt n}.
$$
$T$ is no longer standard normal. Both the numerator and the denominator are random, and they are dependent ($s$ is computed from the same sample as $\bar X$). The correct sampling distribution is Student's $t$ with
$$
df=n-1
$$
degrees of freedom.

Why $n-1$? The vector of residuals $(X_1-\bar X,\dots,X_n-\bar X)$ lives in the hyperplane $\sum_i (X_i-\bar X)=0$. One linear constraint has been used; $n-1$ free directions remain. That is the same lost degree of freedom as Bessel's correction.

The $t$ density has **heavier tails** than $N(0,1)$ when $df$ is small, which makes the critical value $t_{n-1,1-\alpha/2}$ larger than $1.96$ and the interval wider — exactly the extra uncertainty we should pay for estimating $\sigma$. As $n\to\infty$, $s\to\sigma$ and $t_{n-1}\to N(0,1)$.

$$
\boxed{\bar X \pm t_{n-1,\,1-\alpha/2}\,\frac{s}{\sqrt n}}
$$

The $t$ critical value is again an inverse CDF:
$$
\boxed{t_{n-1,1-\alpha/2}=F_{t_{n-1}}^{-1}(1-\alpha/2).}
$$
In the helper this is `stats.t.ppf(1 - α/2, n-1)`. For $n=20$ and $\alpha=0.05$ you will see $t^*\approx 2.093$, larger than $1.96$: the interval pays a "$t$ tax" for estimating $\sigma$.

### Frequentist meaning

The parameter $\mu$ is fixed. The interval is random. After we have computed $[L,U]$ from one sample, $\mu$ is either inside or not: the probability is $0$ or $1$.

> **Crucial Note:** A $95\%$ CI does *not* mean $P(\mu\in[L,U]\mid \text{data})=0.95$. It means: if we repeat the whole procedure, about $95\%$ of the random intervals we construct will contain the fixed $\mu$.

The next experiment is designed so you can *see* that sentence instead of reciting it.


In [ ]:
# Heavier tails at small df; convergence to Normal as df grows.
visualization_utils.plot_t_vs_normal()

# Static 95% coverage picture before the widget.
visualization_utils.plot_repeated_confidence_intervals(n=20, confidence=0.95, num_intervals=60, seed=42)


Blue intervals cover the fixed black line; red ones miss. At $95\%$ we expect about $57/60$ to be blue. They are not all the same width, because $s$ changes from sample to sample — that is the $t$ tax.

Now change $n$ and the confidence level yourself:

* $99\%$ intervals are *wider* than $95\%$ intervals. Higher confidence is bought with more caution, not with magic.
* Increasing $n$ makes intervals *narrower*, through $SE=s/\sqrt n$.
* The black line never moves. The intervals do. That is the frequentist picture.


In [ ]:
# Interactive coverage laboratory.
# Watch the hit rate stay near the nominal confidence, while width tracks n and α.
visualization_utils.interactive_confidence_intervals()


## 9. Hypothesis Testing

Suppose model B records a higher test accuracy than model A. We cannot immediately conclude that B is genuinely better. The observed gap is a statistic, and statistics move.

Hypothesis testing is a formal way to ask: **if nothing interesting were happening, how surprising would this result be?**

* $H_0$ — the **null**: no genuine difference (e.g. $\mu_B-\mu_A=0$)
* $H_a$ — the **alternative**: a genuine difference

We do not "prove" $H_a$. We assume $H_0$ and measure compatibility.

### The $t$-statistic

$$
t=\frac{\text{observed effect}}{\text{standard error of the effect}}.
$$
For a one-sample mean,
$$
\boxed{t=\frac{\bar X-\mu_0}{s/\sqrt n}, \qquad df=n-1.}
$$

* numerator: how far the estimate sits from the null value
* denominator: how far we *expect* it to sit under sampling noise
* large $|t|$: the effect is large relative to uncertainty

The statistic $t$ is **not** a probability. To turn it into a $p$-value we need its **null distribution**.

### $z$ versus $t$

| Situation | Statistic | Reference |
|---|---|---|
| $\sigma$ known | $z=(\bar X-\mu_0)/(\sigma/\sqrt n)$ | $N(0,1)$ |
| $\sigma$ unknown, use $s$ | $t=(\bar X-\mu_0)/(s/\sqrt n)$ | $t_{n-1}$ |
| $n$ large | $t\approx z$ | the $t$ tax vanishes |

For small samples with unknown $\sigma$, using $z$ and $1.96$ understates uncertainty. That is the same picture as the CI section.

### p-values

$$
\boxed{p=P\bigl(\text{result at least as extreme as observed}\ \big|\ H_0\bigr)}
$$

A $p$-value is **not**:

* $P(H_0\mid \text{data})$
* "the probability that this happened by chance"
* the probability that $H_0$ is true
* the probability that $H_a$ is true

It is a tail probability computed *inside* the null model.

We choose a threshold $\alpha$ (often $0.05$) **before** seeing the data.

* $p<\alpha$ $\rightarrow$ **reject** $H_0$
* $p\geq\alpha$ $\rightarrow$ **fail to reject** $H_0$

We do not "accept $H_0$." Absence of evidence is not evidence of absence; the test may simply have been underpowered.

### From a test statistic to a $p$-value: the CDF does the conversion

The key idea is always the same. The test statistic ($t$, $z$, $F$, $\chi^2$, McNemar, \ldots) is converted into a $p$-value by asking:

> If $H_0$ were true, how much probability lies at least as far into the tail(s) of the **null distribution** as the statistic we observed?

The CDF is exactly the tool that computes that probability.

Suppose under $H_0$ the statistic $T$ has a known continuous distribution, for example $T\sim t_{\nu}$. The CDF is
$$
F(t)=P(T\leq t).
$$
Graphically, $F(2)$ is the area under the density to the **left** of $2$. For a continuous distribution the right tail is the complement:
$$
\boxed{P(T>t)=1-F(t).}
$$
Libraries expose this as the **survival function** `sf(t)`, which is numerically more accurate in the far tail than computing `1 - cdf(t)` directly (floating-point cancellation when $F(t)$ is extremely close to $1$).

The universal pipeline is
$$
\boxed{
\text{data}
\rightarrow
\text{test statistic}
\rightarrow
\text{null distribution}
\rightarrow
\text{tail probability (via CDF / SF / PMF)}
\rightarrow
p\text{-value}
}
$$

| Statistic | Null distribution | $p$ (typical) |
|---|---|---|
| $t$ | $t_{\nu}$ | $2\bigl(1-F_{t_\nu}(\|t\|)\bigr)$ two-sided |
| $z$ | $N(0,1)$ | $2\bigl(1-\Phi(\|z\|)\bigr)$ two-sided |
| $\chi^2$ | $\chi^2_{\nu}$ | $1-F_{\chi^2_\nu}(\chi^2)$ (upper tail) |
| $F$ (ANOVA) | $F_{\nu_1,\nu_2}$ | $1-F_{F}(F_{\mathrm{obs}})$ (upper tail) |
| exact McNemar | $\mathrm{Binomial}(n,1/2)$ | sum of PMF in the extreme tails |

The $p$-value is not a "$t$-test thing." It is a tail probability of **whatever statistic you used**.

### Worked example: $t_{\mathrm{obs}}=2.3$, $df=19$

Under $H_0:\mu=\mu_0$ we have $T\sim t_{19}$. Suppose $F_{t_{19}}(2.3)\approx 0.983$. Then $1-F(2.3)\approx 0.017$. The alternative decides *which* tail(s) count as "extreme."

**Right-tailed** ($H_1:\mu>\mu_0$). Unusually *large* values are evidence against $H_0$:
$$
\boxed{p=P(T\geq 2.3)=1-F_{t_{19}}(2.3)\approx 0.017.}
$$
In code: `stats.t.sf(2.3, 19)`.

**Left-tailed** ($H_1:\mu<\mu_0$), still with $t=2.3$. Now $2.3$ is *not* extreme on the left:
$$
p=P(T\leq 2.3)=F_{t_{19}}(2.3)\approx 0.983.
$$
If instead we had observed $t=-2.3$, then $p=F_{t_{19}}(-2.3)\approx 0.017$.

**Two-sided** ($H_1:\mu\neq\mu_0$). Both directions count, so
$$
p=P(T\geq 2.3)+P(T\leq -2.3).
$$
The $t$ distribution is symmetric, $P(T\geq 2.3)=P(T\leq -2.3)$, and therefore
$$
\boxed{p=2\bigl[1-F_{t_{19}}(|t_{\mathrm{obs}}|)\bigr]\approx 2\times 0.017=0.034.}
$$
The absolute value says: we care about *distance from zero*, not the sign. This is exactly `2 * stats.t.sf(abs(t_obs), df)` in `_t_pvalue`.

| Alternative | Formula | $p$ (this example) |
|---|---|---:|
| $H_1:\mu>\mu_0$ | $1-F(t)$ | $0.017$ |
| $H_1:\mu<\mu_0$ | $F(t)$ | $0.983$ |
| $H_1:\mu\neq\mu_0$ | $2\bigl(1-F(|t|)\bigr)$ | $0.034$ |

> **Crucial distinction:** You cannot compute a $p$-value from $t$ alone. You need (1) the observed statistic, (2) its null distribution / degrees of freedom, and (3) whether the test is one- or two-sided.

Using a one-sided test after seeing the sign of the gap is just $\alpha$-hacking. For model comparison, both improvement *and* degradation usually matter, so the default is two-sided unless a directional hypothesis was committed to in advance.

### The critical region

A $p$-value is the continuous version of a **critical-region** rule. Fix $\alpha$ *before* seeing the data. The critical region $C$ is the set of statistic values that lead to rejection.

For a **right-tailed $z$-test** at $\alpha=0.05$,
$$
Z_c=\Phi^{-1}(1-\alpha)=\Phi^{-1}(0.95)\approx 1.645.
$$
$$
\boxed{C=\{z:z\geq 1.645\}.}
$$
On the sample-mean scale, with known $\sigma$,
$$
C=\Bigl\{\bar x:\bar x\geq \mu_0+1.645\,\frac{\sigma}{\sqrt n}\Bigr\}.
$$
The number $1.645$ is **not** magic: it is the $z$ that leaves probability $\alpha=0.05$ in the right tail of $N(0,1)$. For a two-sided test the tail is split, so
$$
Z_c=\Phi^{-1}(1-\alpha/2)\approx 1.960,
$$
and we reject when $|Z|\geq 1.960$. For a $t$-test, replace $\Phi^{-1}$ by $F_{t_{n-1}}^{-1}$ (heavier tails, larger $|t_c|$ at small $n$).

$p<\alpha$ if and only if the observed statistic lands in $C$. Equivalently, $p$ is the smallest $\alpha$ for which the observed result would have been rejected.

The pictures below shade $C$ and the $p$-value region for the same observed $t$.


In [ ]:
# The same observed t can sit inside the two-tailed acceptance region
# and outside the one-tailed rejection region.
visualization_utils.plot_one_vs_two_tailed(t_obs=1.85, df=20, alpha=0.05)
visualization_utils.plot_pvalue_region(t_obs=2.40, df=19)

# CDF → tail probability → p-value, for t_obs = 2.3, df = 19.
cdf_demo = visualization_utils.plot_cdf_to_pvalue(t_obs=2.3, df=19)
print("F(2.3) =", round(cdf_demo["F"], 4))
print("right-tailed p = 1-F     =", round(cdf_demo["p_right"], 4))
print("left-tailed  p = F       =", round(cdf_demo["p_left"], 4))
print("two-sided    p = 2(1-F)  =", round(cdf_demo["p_two"], 4))

# Critical region: Zc ≈ 1.645 (one-sided α=0.05) and ±1.96 (two-sided).
crit = visualization_utils.plot_critical_region(alpha=0.05, n=30, sigma=1.0)
print(f"Zc one-sided  = {crit['z_one']:.3f}   (inverse Phi at 0.95)")
print(f"Zc two-sided  = {crit['z_two']:.3f}   (inverse Phi at 0.975)")
print(f"Reject H0 (one-sided) if X̄ > {crit['xbar_critical_one']:.3f}")


### Experiment: p-values under a true null

If $H_0$ is true and the test is valid and continuous, $p$ is Uniform$(0,1)$. In particular, about $5\%$ of honest tests reject at $\alpha=0.05$. That is not a malfunction. That *is* Type I error.


In [ ]:
# 10,000 honest one-sample t-tests under H0: μ = 0.
rng = np.random.default_rng(42)
p_values = []
for _ in range(10_000):
    sample = rng.normal(0.0, 1.0, 30)
    _, p, _ = hypothesis_testing.one_sample_t_test(sample, 0.0)
    p_values.append(p)

type_I = visualization_utils.plot_null_pvalues(p_values, alpha=0.05)
print(f"Empirical Type I rate = {type_I:.4f}  (target 0.05)")


The histogram is flat. About one in twenty tests of a *true* null crosses $0.05$. If you run enough comparisons, some of them will look "significant" even when nothing is happening. That fact will return when we refuse to replace ANOVA by a pile of pairwise $t$-tests.

## 10. Type I Error, Type II Error, and Power

### Type I error
Rejecting $H_0$ when $H_0$ is true. Long-run probability: $\alpha$. This is exactly the area of the critical region *under the null*.

### Type II error
Failing to reject $H_0$ when $H_0$ is false. Probability: $\beta$. This is the area of the *non*-rejection region under the alternative.

### Power
$$
\boxed{Power=1-\beta=P(\text{reject }H_0\mid H_1\text{ is true})}
$$
If a real effect exists, how likely is the test to detect it?

### Why a larger effect raises power: signal versus noise

Write the true mean under $H_1$ as $\mu_1=\mu_0+\delta$. The **effect size** $\delta$ is the signal. The sampling noise of the mean is the standard error $\sigma/\sqrt n$. Their ratio is how distinguishable the two sampling distributions are:
$$
\boxed{\frac{\text{signal}}{\text{noise of }\bar X}=\frac{\delta}{\sigma/\sqrt n}=d\sqrt n,\qquad d=\frac{\delta}{\sigma}.}
$$
$d$ is Cohen's $d$: the difference measured in population standard deviations. Noise here *is* variation — the spread $\sigma$ of individual observations. Larger $\sigma$ buries the same $\delta$ in a wider sampling distribution of $\bar X$; larger $n$ shrinks that width like $1/\sqrt n$.

For a right-tailed $z$-test the critical cutoff on the $Z$ scale is $Z_c=z_{1-\alpha}\approx 1.645$ when $\alpha=0.05$. Under $H_1$,
$$
Z=\frac{\bar X-\mu_0}{\sigma/\sqrt n}\sim N(d\sqrt n,\,1),
$$
so
$$
\boxed{\text{Power}=\Phi\!\bigl(d\sqrt n-Z_c\bigr)=P\bigl(\bar X > \mu_0+Z_c\,\sigma/\sqrt n \bigm| \mu=\mu_1\bigr).}
$$
The formula makes the four knobs obvious:

* larger $|\delta|$ (or $d$) $\rightarrow$ $H_0$ and $H_1$ sampling distributions overlap less $\rightarrow$ more of $H_1$ sits past the critical line $\rightarrow$ higher power
* larger $\sigma$ (more noise) $\rightarrow$ more overlap $\rightarrow$ lower power
* larger $n$ $\rightarrow$ both curves get thinner and the alternative's centre is $d\sqrt n$ standard errors from $\mu_0$ $\rightarrow$ higher power
* larger $\alpha$ $\rightarrow$ $Z_c$ moves toward zero (easier rejection) $\rightarrow$ higher power **and** more Type I error

There is no free lunch in that last knob: you are simply declaring more false alarms acceptable.

The laboratory uses a normal approximation to the sampling distribution of $\bar X$ (valid by the CLT, and exact if the data are Gaussian with known $\sigma$). The code computes
`crit = norm.ppf(1-α, loc=0, scale=SE)` and `power = 1 - norm.cdf(crit, loc=δ, scale=SE)`, which *is* $\Phi(d\sqrt n-Z_c)$.

The next laboratory lets you move those four knobs and watch the overlap change.


In [ ]:
# Interactive power laboratory.
# The blue tail past the red line is Type I; the orange mass left of the line is Type II.
visualization_utils.interactive_power_experiment()

# Quantitative summary of the same four relationships.
visualization_utils.plot_power_relationships()


A small effect buried in noise with $n=10$ is almost invisible. The same effect with $n=160$, or a larger effect with the original $n$, lights up. Raising $\alpha$ from $0.01$ to $0.10$ also raises power; you are simply declaring more false alarms acceptable.

## 11. $t$-tests in Practice

### One-sample $t$-test

Compare one group's mean to a fixed theoretical value $\mu_0$. We compute every piece from the formula
$$
t=\frac{\bar X-\mu_0}{s/\sqrt n},\qquad
p_{\text{two-sided}}=2\bigl(1-F_{t_{n-1}}(|t|)\bigr)=2\cdot\mathrm{sf}(|t|;\,n-1),
$$
then ask SciPy for the same numbers. The library is a check, not the explanation. The CI uses the inverse CDF,
$$
t^*=F_{t_{n-1}}^{-1}(1-\alpha/2),\qquad \bar X\pm t^*\,\frac{s}{\sqrt n}.
$$


In [ ]:
# Manual one-sample t-test vs SciPy.
rng = np.random.default_rng(0)
sample = rng.normal(loc=0.45, scale=1.0, size=25)
mu0 = 0.0

n = len(sample)
xbar = estimator_utils.sample_mean(sample)
s = np.sqrt(estimator_utils.unbiased_variance(sample))
se = s / np.sqrt(n)
t_by_hand = (xbar - mu0) / se
df = n - 1
F_abs = stats.t.cdf(abs(t_by_hand), df)
p_via_cdf = 2 * (1 - F_abs)
p_via_sf = 2 * stats.t.sf(abs(t_by_hand), df)
low, high, _, _, t_crit = hypothesis_testing.one_sample_t_ci(sample, 0.95)

cmp = hypothesis_testing.compare_one_sample_with_scipy(sample, mu0)
print(f"n={n},  X̄={xbar:.4f},  s={s:.4f},  SE={se:.4f},  df={df}")
print(f"t = (X̄ - μ0) / (s/√n) = {t_by_hand:.6f}")
print(f"F(|t|) = {F_abs:.6f}   →   2(1-F) = {p_via_cdf:.6f}")
print(f"2·sf(|t|)            = {p_via_sf:.6f}   (preferred numerically)")
print(f"t from SciPy  = {cmp['scipy_t']:.6f}   |gap|={cmp['t_gap']:.2e}")
print(f"p from SciPy  = {cmp['scipy_p']:.6f}   |gap vs sf|={abs(p_via_sf-cmp['scipy_p']):.2e}")
print(f"t* = inverse t-CDF at 0.975 (df={df}) = {t_crit:.3f}")
print(f"95% CI = X̄ ± t*·SE = [{low:.4f}, {high:.4f}]")
print("Decision:", "reject H0" if cmp['manual_p'] < 0.05 else "fail to reject H0")


Any remaining gap is floating-point bookkeeping, not a different test. The module uses `stats.t.sf` rather than `1 - stats.t.cdf` so tiny tail probabilities do not collapse to $0$ from rounding.

### Two-sample tests: pooled variance versus Welch

To compare two independent groups we still form
$$
t=\frac{\bar X_A-\bar X_B}{SE(\bar X_A-\bar X_B)}.
$$
Independence gives $\mathrm{Var}(\bar X_A-\bar X_B)=\sigma_A^2/n_A+\sigma_B^2/n_B$, so the only modelling choice is how to estimate that variance.

If we are willing to assume a common variance $\sigma_A^2=\sigma_B^2=\sigma^2$, the **pooled** estimator is the degrees-of-freedom-weighted average of the two sample variances:
$$
s_p^2
=
\frac{(n_A-1)s_A^2+(n_B-1)s_B^2}{n_A+n_B-2}.
$$
(The $n_j-1$ weights are the same Bessel counts that make each $s_j^2$ unbiased.) Then
$$
SE=s_p\sqrt{\frac{1}{n_A}+\frac{1}{n_B}},
\qquad
df=n_A+n_B-2,
\qquad
t=\frac{\bar X_A-\bar X_B}{SE}.
$$
Blindly assuming $\sigma_A=\sigma_B$ is a strong claim.

**Welch's test** does not make that claim. Plug the two sample variances directly into the independent-sum formula:
$$
SE=\sqrt{\frac{s_A^2}{n_A}+\frac{s_B^2}{n_B}},
\qquad
t=\frac{\bar X_A-\bar X_B}{SE}.
$$
The exact null distribution is no longer $t$ with integer $df$. Satterthwaite's approximation matches the first two moments of a linear combination of $\chi^2$ random variables and yields
$$
\boxed{
\nu=\frac{\bigl(s_A^2/n_A+s_B^2/n_B\bigr)^2}{(s_A^2/n_A)^2/(n_A-1)+(s_B^2/n_B)^2/(n_B-1)}.
}
$$
$\nu$ is typically not an integer; the $t_{\nu}$ CDF is still well-defined. Welch is the default for independent ML-metric comparisons unless equal variance is actually justified.


In [ ]:
# Unequal-variance world: Welch vs pooled vs SciPy.
rng = np.random.default_rng(1)
group_a = rng.normal(0.0, 1.0, 18)    # small variance
group_b = rng.normal(0.8, 2.5, 40)    # larger mean *and* larger variance

t_w, p_w, df_w = hypothesis_testing.two_sample_t_test_welch(group_a, group_b)
t_p, p_p, df_p, sp2 = hypothesis_testing.pooled_two_sample_t_test(group_a, group_b)
welch_cmp = hypothesis_testing.compare_welch_with_scipy(group_a, group_b)

print("Welch (preferred when variances may differ)")
print(f"  t={t_w:.4f}, p={p_w:.4f}, df={df_w:.2f}")
print(f"  SciPy t={welch_cmp['scipy_t']:.4f}, p={welch_cmp['scipy_p']:.4f}   |t gap|={welch_cmp['t_gap']:.2e}")
print()
print("Pooled (assumes σ_A = σ_B)")
print(f"  t={t_p:.4f}, p={p_p:.4f}, df={df_p},  s_p²={sp2:.3f}")
print()
print(f"Sample variances: s_A²={np.var(group_a, ddof=1):.3f}, s_B²={np.var(group_b, ddof=1):.3f}")
print("The two tests disagree on df and can disagree on the p-value when variances differ.")


### Paired $t$-test

Sometimes observations come in pairs: before/after, same subject under two conditions, same test example scored by two models on a *continuous* metric.

Define $D_i=X_{B,i}-X_{A,i}$. The paired test is a one-sample test on the differences:
$$
t=\frac{\bar D}{s_D/\sqrt n}, \qquad H_0:\mu_D=0.
$$
Pairing cancels the shared baseline. A large patient-to-patient (or example-to-example) offset that appears in *both* measurements disappears in $D_i$, so $s_D$ can be much smaller than the two-sample standard error. Treating paired data as independent throws that information away and loses power.


In [ ]:
# Same 30 people, +3 point training effect, large between-person variance.
rng = np.random.default_rng(42)
baseline = rng.normal(50, 15, 30) # sd is large intentionally so that welch will miss it 
after = baseline + rng.normal(3, 2, 30)

t_ind, p_ind, _ = hypothesis_testing.two_sample_t_test_welch(after, baseline)
t_pair, p_pair, _ = hypothesis_testing.paired_t_test(after, baseline)
scipy_pair = stats.ttest_rel(after, baseline)

visualization_utils.plot_paired_vs_independent(baseline, after)
print(f"Independent Welch: t={t_ind:.3f}, p={p_ind:.4f}   (often misses the +3 effect)")
print(f"Paired t-test:     t={t_pair:.3f}, p={p_pair:.4f}")
print(f"SciPy ttest_rel:   t={scipy_pair.statistic:.3f}, p={scipy_pair.pvalue:.4f}   |t gap|={abs(t_pair-scipy_pair.statistic):.2e}")


The left panel shows why pairing matters: every grey line is a person. The vertical scatter is huge; the *lifts* are small and consistent. The right panel is the distribution the paired test actually uses.

### Assumptions of the classical $t$-test

* **Independence.** The intended sampling unit must not be secretly clustered (same user many times, leaking train/test, etc.).
* **Continuous outcome.** Means of $0/1$ correctness are an approximation; McNemar is the paired binary tool.
* **Distribution.** For small $n$, a violent departure from normality can distort $p$-values. For larger $n$ the CLT makes the test much more robust. The test does *not* require perfectly Gaussian data in all circumstances.

The next simulation keeps $H_0$ true on exponential data and watches Type I error at small versus large $n$.


In [ ]:
# H0 is true (exponential mean = 1). Does the t-test still have Type I rate ≈ 0.05?
rates = visualization_utils.plot_ttest_assumption_simulation(
    n_small=8, n_large=80, n_reps=4000, seed=2, alpha=0.05
)
print(rates)


At $n=8$ the $p$-value histogram is a little warped and the Type I rate drifts. At $n=80$ it is close to uniform again. Non-normality is a small-sample problem first.

## 12. McNemar's Test for Paired Classifiers

A paired $t$-test on binary correctness is the wrong instrument. Two classifiers evaluated on the *same* test examples produce paired Bernoulli outcomes, not continuous measurements.

McNemar's test is built for that $2\times 2$ table:

|  | B correct | B wrong |
|---|---|---|
| A correct | $a=N_{11}$ (both right) | $b=N_{10}$ (A only) |
| A wrong | $c=N_{01}$ (B only) | $d=N_{00}$ (both wrong) |

Agreements $a$ and $d$ do not tell us which model is better. McNemar uses only the **discordant** pairs $b,c$, because those are the trials on which the two methods disagree.

Under $H_0$ the two methods are equally likely to "win" a discordant trial, so
$$
H_0:p=\tfrac12
$$
for each discordant observation.

### Asymptotic $\chi^2$ statistic

The large-sample McNemar statistic is
$$
\boxed{\chi^2=\frac{(b-c)^2}{b+c}}
$$
and with Edwards' continuity correction (the default in the helper)
$$
\chi^2=\frac{(|b-c|-1)^2}{b+c}.
$$
Under $H_0$, approximately $\chi^2\sim\chi^2_1$. Unlike $t$, a chi-square random variable is supported only on $[0,\infty)$, so "at least as extreme" means **large**:
$$
\boxed{p=P(\chi^2_1\geq \chi^2_{\mathrm{obs}})=1-F_{\chi^2_1}(\chi^2_{\mathrm{obs}})=\mathrm{sf}(\chi^2_{\mathrm{obs}};1).}
$$

Worked numbers: $b=30$, $c=10$ (no continuity correction) give
$$
\chi^2=\frac{(30-10)^2}{40}=10,
\qquad
p=1-F_{\chi^2_1}(10)\approx 0.0016.
$$

### Connection to a two-sided $z$-test

A $\chi^2_1$ random variable is a squared standard normal:
$$
\boxed{Z^2\sim\chi^2_1.}
$$
So $\chi^2=10$ corresponds to $|Z|=\sqrt{10}\approx 3.162$, and
$$
p=2\,P(Z\geq 3.162)=2\bigl(1-\Phi(3.162)\bigr)\approx 0.0016.
$$
Same answer. McNemar's $\chi^2$ *is* the two-sided normal test of $H_0:p=1/2$ on the discordant pairs, written in squared form.

### Exact binomial McNemar

The $\chi^2_1$ approximation needs a reasonably large $b+c$. The exact test does not. Let $n=b+c$ and let $X$ be the number of times A wins among discordant pairs. Under $H_0$,
$$
X\sim\mathrm{Binomial}(n,1/2),
\qquad
P(X=k)=\binom{n}{k}\Bigl(\tfrac12\Bigr)^n.
$$
For $b=30$, $c=10$ we have $n=40$ and $X=30$. The upper tail is
$$
P(X\geq 30)=\sum_{k=30}^{40}\binom{40}{k}(1/2)^{40}.
$$
A symmetric two-sided exact $p$-value doubles the more extreme tail (and clips at $1$):
$$
p=2\,P\bigl(X\geq \max(b,c)\bigr).
$$
The idea has not changed:
$$
\boxed{p=\text{probability of outcomes at least as extreme as observed, under }H_0.}
$$
The only difference is whether we **integrate a density** (CDF of $t$, $z$, $\chi^2$, $F$) or **sum a PMF** (binomial).

This is a small but important statistical distinction, not a full classifier-evaluation course.


In [ ]:
# Shared test set of 200 examples. B is genuinely a bit better, with both kinds of discord.
rng = np.random.default_rng(3)
n = 200
y_true = np.ones(n, dtype=int)
latent = rng.random(n)
correct_a = (latent < 0.78) ^ (rng.random(n) < 0.04)
correct_b = (latent < 0.88) ^ (rng.random(n) < 0.04)
y_pred_a = correct_a.astype(int)
y_pred_b = correct_b.astype(int)

chi2, p_mc, table = hypothesis_testing.mcnemars_test(y_true, y_pred_a, y_pred_b)
visualization_utils.plot_mcnemar_table(table)
print(f"Accuracy A = {correct_a.mean():.3f},  Accuracy B = {correct_b.mean():.3f}")
print(f"Discordant: b=N10={table['a_only']}, c=N01={table['b_only']}")
print(f"McNemar χ² (continuity-corrected) = {chi2:.3f}, p = {p_mc:.4f}")

# Textbook numbers from the note: b=30, c=10.
demo = visualization_utils.plot_mcnemar_null(n10=30, n01=10)
print("Worked example b=30, c=10 (no continuity correction)")
print(f"  χ² = (b-c)²/(b+c) = {demo['chi2']:.1f}")
print(f"  asymptotic p = 1 - F_χ²_1(10) = {demo['p_asym']:.4f}")
print(f"  two-sided normal p from |Z|=√10 = {demo['p_from_z']:.4f}")
print(f"  exact binomial p = {demo['p_exact']:.4f}")

# Naive (wrong) move: treat 0/1 correctness as independent continuous data.
t_wrong, p_wrong, _ = hypothesis_testing.two_sample_t_test_welch(
    correct_a.astype(float), correct_b.astype(float)
)
print(f"Naive Welch on 0/1 correctness: t={t_wrong:.3f}, p={p_wrong:.4f}")
print("That test ignores pairing and treats binary values as continuous.")


## 13. Confidence Intervals $\leftrightarrow$ Tests, and Practical Significance

A two-sided test at level $\alpha$ and a $1-\alpha$ confidence interval are the same information:

* $\mu_0$ **inside** the $95\%$ CI $\;\Longleftrightarrow\;$ fail to reject $H_0:\mu=\mu_0$ at $\alpha=0.05$
* $\mu_0$ **outside** the CI $\;\Longleftrightarrow\;$ reject $H_0$

Why? The two-sided $t$-test rejects when $|t|\geq t_{n-1,1-\alpha/2}$, i.e. when
$$
\bigl|\bar X-\mu_0\bigr|\geq t^*\,\frac{s}{\sqrt n},
$$
which is exactly the statement that $\mu_0$ lies outside $\bar X\pm t^* s/\sqrt n$. The CI is the more informative object: it shows the plausible *size* of the effect, not only a binary decision.

**Statistical significance is not practical significance.**

* A $0.001$ accuracy gap becomes $p<0.05$ if $n$ is huge: the SE $s/\sqrt n$ is tiny, so even a negligible numerator produces a large $t$.
* A $12$ point gap can fail to reach $p<0.05$ if $n$ is tiny and noisy: the signal is real-looking but the noise of $\bar X$ is too large to distinguish it from $0$.

$p<0.05$ does not mean the effect is important. Look at the estimate and the interval, not only the star.


In [ ]:
# Same logic, two pictures: CI vs test, then tiny-but-significant vs big-but-not.
visualization_utils.plot_ci_vs_hypothesis(sample_mean=0.0, se=0.30, null_value=0.0, alpha=0.05)
practical = visualization_utils.plot_practical_vs_statistical()
print(practical)


## 14. ML Model-Comparison Laboratory

We now put the pieces on a model-comparison question:

> Model B's measured accuracy is higher than model A's. Can we reliably say B is better?

We simulate two classifiers on repeated draws of a test set, record $\Delta=\mathrm{acc}_B-\mathrm{acc}_A$, and look at the sampling distribution of $\Delta$, a confidence interval, a paired test, and McNemar on one draw.

On a shared test set of size $n$, the accuracy difference is completely determined by the discordant counts:
$$
\widehat\Delta=\mathrm{acc}_B-\mathrm{acc}_A=\frac{c-b}{n}=\frac{N_{01}-N_{10}}{n}.
$$
Agreements cancel. A Wald standard error for this paired-proportion difference uses
$$
\widehat{SE}(\widehat\Delta)\approx\sqrt{\frac{\hat p_{10}+\hat p_{01}}{n}}=\sqrt{\frac{b+c}{n^2}},
$$
because each example contributes a $\{-1,0,+1\}$ difference whose variance is (to first order) the discordant rate $(b+c)/n$. The $95\%$ Wald interval is then
$$
\widehat\Delta\pm 1.96\,\widehat{SE}(\widehat\Delta).
$$
If $0$ is outside that interval, the two-sided normal test of $H_0:\Delta=0$ rejects — the same logic as McNemar, in CI clothing.

This is a statistical reasoning experiment, not a training project.


In [ ]:
def simulate_paired_classifiers(n=350, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.ones(n, dtype=int)
    latent = rng.random(n)
    # B is slightly better; independent flips create both kinds of discordant pairs.
    correct_a = (latent < 0.80) ^ (rng.random(n) < 0.05)
    correct_b = (latent < 0.86) ^ (rng.random(n) < 0.05)
    return y_true, correct_a.astype(int), correct_b.astype(int)


# One observed test set
y_true, pred_a, pred_b = simulate_paired_classifiers(seed=10)
obs_diff = pred_b.mean() - pred_a.mean()
chi2, p_mc, table = hypothesis_testing.mcnemars_test(y_true, pred_a, pred_b)

# Repeated test-set draws: sampling variability of the accuracy gap
diffs = []
for i in range(300):
    _, a, b = simulate_paired_classifiers(seed=100 + i)
    diffs.append(b.mean() - a.mean())
diffs = np.asarray(diffs)
# Wald CI for the paired accuracy difference uses only discordant pairs.
n = len(pred_a)
# Wald SE for paired proportion difference ≈ sqrt((p10+p01)/n)
p10, p01 = table["a_only"] / n, table["b_only"] / n
se_wald = np.sqrt((p10 + p01) / n)
ci_wald = (obs_diff - 1.96 * se_wald, obs_diff + 1.96 * se_wald)

visualization_utils.plot_model_comparison(diffs, obs_diff, ci_wald, p_mc)
print(f"Observed acc A={pred_a.mean():.3f}, acc B={pred_b.mean():.3f}, Δ={obs_diff:.3f}")
print(f"McNemar: χ²={chi2:.3f}, p={p_mc:.4f}")
print(f"95% Wald CI for Δ: [{ci_wald[0]:.3f}, {ci_wald[1]:.3f}]")
print(f"Mean Δ across 300 test-set draws: {diffs.mean():.3f}  (SD {diffs.std(ddof=1):.3f})")
print("A single lucky test set can exaggerate Δ; the CI and McNemar keep us honest.")


## 15. Analysis of Variance (ANOVA)

With three or more groups, "just run every pairwise $t$-test" is a bad plan. The number of pairs is $\binom{k}{2}$, and each test has its own Type I chance. If those tests were independent, the **family-wise error rate** (probability of at least one false rejection) would be
$$
\boxed{\mathrm{FWER}=1-(1-\alpha)^{m},\qquad m=\binom{k}{2}.}
$$
For $k=4$ groups, $m=6$ pairs, $\alpha=0.05$: $\mathrm{FWER}\approx 0.26$, not $0.05$. The tests are not quite independent, but the message stands: uncorrected pairwise $t$-tests inflate false discoveries. ANOVA asks a *single* question first:
$$
H_0:\mu_1=\mu_2=\cdots=\mu_k
\qquad\text{versus}\qquad
H_a:\text{at least one mean differs}.
$$

> **Crucial distinction:** A significant ANOVA does **not** identify which groups differ. It only says the means are not all equal.

### Within-group versus between-group variation

ANOVA compares two kinds of variation:

* **within-group** — how noisy each group is around its own mean
* **between-group** — how far the group means sit from the grand mean

Large between-group variation *relative to* within-group variation is evidence against $H_0$. That is the same signal-versus-noise idea as power: $MS_B$ is the signal among means, $MS_W$ is the noise.

### Sum-of-squares decomposition

Write each observation as a grand-mean deviation split into two pieces:
$$
X_{ij}-\bar X
=
(X_{ij}-\bar X_j)
+
(\bar X_j-\bar X).
$$
Square both sides and sum over $i,j$. The cross term is
$$
\sum_j\sum_i (X_{ij}-\bar X_j)(\bar X_j-\bar X)
=
\sum_j(\bar X_j-\bar X)\sum_i(X_{ij}-\bar X_j)
=
\sum_j(\bar X_j-\bar X)\cdot 0
=
0,
$$
because the residuals inside a group sum to zero. What remains is
$$
\boxed{SS_T=SS_W+SS_B}
$$
with
$$
SS_W=\sum_j\sum_i(X_{ij}-\bar X_j)^2,
\qquad
SS_B=\sum_j n_j(\bar X_j-\bar X)^2.
$$
The factor $n_j$ is not decoration: a group with more observations contributes more to the between-group total. $SS_T=\sum_{i,j}(X_{ij}-\bar X)^2$ is the usual total sum of squares.

### Mean squares and the $F$ statistic

We convert sums of squares into *mean* squares by dividing by the appropriate degrees of freedom.

* Between groups: $k$ means, one constraint (they average to $\bar X$ in the weighted sense) $\rightarrow$ $df_B=k-1$
* Within groups: $N$ observations, $k$ estimated group means $\rightarrow$ $df_W=N-k$

$$
MS_B=\frac{SS_B}{k-1},
\qquad
MS_W=\frac{SS_W}{N-k},
\qquad
\boxed{F=\frac{MS_B}{MS_W}}.
$$

Under $H_0$ (and Gaussian errors, equal variance) both mean squares estimate the same $\sigma^2$, so $F\sim F_{k-1,\,N-k}$ and $F\approx 1$. If the group means really differ, $MS_B$ inflates and $F\gg 1$. The test is **upper-tailed**:
$$
\boxed{p=P(F_{k-1,N-k}\geq F_{\mathrm{obs}})=1-F_{\mathrm{CDF}}(F_{\mathrm{obs}})=\mathrm{sf}(F_{\mathrm{obs}};\,k-1,N-k).}
$$
There is no left tail to worry about: $F<1$ is *more* compatible with $H_0$, not evidence against it.


In [ ]:
# Four designed worlds: identical means, separated means, noisy, both.
visualization_utils.plot_anova_intuition(
    [
        {"name": "Almost identical means", "means": [0.0, 0.15, -0.1], "within_std": 1.0, "n": 25},
        {"name": "Separated means", "means": [-2.0, 0.0, 2.0], "within_std": 1.0, "n": 25},
        {"name": "Separated + noisy", "means": [-2.0, 0.0, 2.0], "within_std": 3.0, "n": 25},
        {"name": "Tiny n, clear gap", "means": [-2.0, 0.0, 2.0], "within_std": 1.0, "n": 8},
    ],
    seed=4,
)


Separation raises $F$. Noise lowers $F$. That is power, again, in ANOVA clothing.

The interactive laboratory lets you move the three means, the within-group standard deviation, and the group size, and watch $SS_B$, $SS_W$, $F$, and $p$ move together.


In [ ]:
visualization_utils.interactive_anova()


### Manual ANOVA, SciPy check, and Tukey HSD

We compute every piece by hand on a controlled dataset, verify $SS_T=SS_B+SS_W$, and compare $F$ and $p$ with SciPy.

A significant ANOVA still does not name the pair. **Tukey's honest significant difference (HSD)** is a post-hoc procedure that tests *every* pair while keeping the family-wise error rate at $\alpha$.

### The studentized range

For $k$ independent $N(\mu,\sigma^2)$ samples of size $n$, the **studentized range** is
$$
Q=\frac{\bar X_{\max}-\bar X_{\min}}{s/\sqrt n},
$$
where $s=\sqrt{MS_W}$ estimates $\sigma$. Under a complete null (all means equal), $Q$ has a known distribution $Q_{k,\,df_W}$ that already accounts for looking at the *largest* of the $\binom{k}{2}$ gaps. The critical value $q^*=q_{\alpha,k,N-k}$ is the upper $\alpha$ quantile of that distribution: `stats.studentized_range.ppf(1-α, k, df_W)`.

For a single pair $(i,j)$ the observed studentized range (Tukey–Kramer, allowing unequal $n$) is
$$
\boxed{
q_{ij}
=
\frac{|\bar X_i-\bar X_j|}{\sqrt{MS_W\cdot\frac12\bigl(\tfrac1{n_i}+\tfrac1{n_j}\bigr)}}.
}
$$
When $n_i=n_j=n$ this collapses to the textbook form $|\bar X_i-\bar X_j|/\sqrt{MS_W/n}$. Reject $H_0:\mu_i=\mu_j$ if
$$
q_{ij}>q_{\alpha,k,N-k}.
$$
The pairwise $p$-value is the survival function of the studentized range, $P(Q_{k,df_W}\geq q_{ij})$, **not** a raw two-sample $t$ $p$-value. That is why Tukey is stricter than an uncorrected $t$-test: $q^*$ is calibrated for the whole family.

### Tukey intervals and the "$0$ inside?" rule

The same comparison can be written as a simultaneous confidence interval for the difference of means:
$$
\boxed{
(\bar X_i-\bar X_j)
\pm
q_{\alpha,k,N-k}
\sqrt{MS_W\cdot\frac12\Bigl(\frac1{n_i}+\frac1{n_j}\Bigr)}.
}
$$
This is the object `pairwise_tukeyhsd` prints as `meandiff`, `lower`, `upper`.

**How to read one row:**

1. `meandiff` is $\bar X_j-\bar X_i$ (the point estimate of $\mu_j-\mu_i$).
2. `[lower, upper]` is the Tukey interval that would cover the true difference in $1-\alpha$ of repeated experiments, *simultaneously for every pair*.
3. Look at whether **$0$ sits inside that interval**.
   * **$0$ inside** $\Rightarrow$ a difference of zero is still compatible with the data at family-wise level $\alpha$ $\Rightarrow$ **fail to reject** $\mu_i=\mu_j$ (`reject = False`).
   * **$0$ outside** $\Rightarrow$ even after paying the multiplicity tax, the interval does not include "no difference" $\Rightarrow$ **reject** that pair (`reject = True`).

That is the same CI $\leftrightarrow$ test duality as Section 13, with a larger critical value than $t_{1-\alpha/2}$ because we are protecting the whole family.

If $k=3$ and only G2 is shifted, you should see: G1 vs G2 red (0 outside), G2 vs G3 red, G1 vs G3 blue (0 inside). ANOVA said "not all equal"; Tukey names the pairs.

### FWER versus Bonferroni

Bonferroni tests each of $m$ pairs at $\alpha/m$. It controls FWER but is conservative (the intervals are wider than necessary). Tukey uses the actual distribution of the *maximum* pairwise gap under normality and equal variance, so it is more powerful than Bonferroni for the all-pairs mean problem while still targeting $\mathrm{FWER}=\alpha$. We compute the Tukey arithmetic from $q^*$ ourselves and check it against `statsmodels`.


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

rng = np.random.default_rng(42)
g1 = rng.normal(1.0, 1.0, 20)
g2 = rng.normal(2.6, 1.0, 20)  # the different group
g3 = rng.normal(0.9, 1.0, 20)

scratch, scipy_cmp = hypothesis_testing.compare_anova_with_scipy(g1, g2, g3)

print("Component          Scratch")
print(f"group means        {scratch['group_means']}")
print(f"overall mean       {scratch['overall_mean']:.4f}")
print(f"SS_W               {scratch['SS_W']:.4f}")
print(f"SS_B               {scratch['SS_B']:.4f}")
print(f"SS_T               {scratch['SS_T']:.4f}")
print(f"SS_B + SS_W        {scratch['SS_B'] + scratch['SS_W']:.4f}   (must match SS_T)")
print(f"df_B, df_W         {scratch['df_B']}, {scratch['df_W']}")
print(f"MS_B, MS_W         {scratch['MS_B']:.4f}, {scratch['MS_W']:.4f}")
print(f"F = MS_B / MS_W    {scratch['F_stat']:.4f}")
print(f"p = P(F ≥ F_obs)   {scratch['p_val']:.4f}")
print()
print(f"SciPy F={scipy_cmp['scipy_F']:.4f}, p={scipy_cmp['scipy_p']:.4f}   "
      f"|F gap|={scipy_cmp['F_gap']:.2e}, |p gap|={scipy_cmp['p_gap']:.2e}")

tukey = hypothesis_testing.tukey_hsd_pairs(g1, g2, g3, names=["G1", "G2", "G3"])
print()
print(f"Tukey q* = F_Q^{{-1}}(1-α; k={tukey['k']}, df={tukey['df_W']}) = {tukey['q_crit']:.4f}")
print(f"Uncorrected FWER if we ran {len(tukey['pairs'])} independent t-tests at 0.05: "
      f"{hypothesis_testing.family_wise_error_rate(0.05, len(tukey['pairs'])):.3f}")
print()
print(f"{'pair':<10} {'diff':>8} {'lower':>8} {'upper':>8} {'0 in CI?':>10} {'reject?':>8} {'p_adj':>8}")
for p in tukey["pairs"]:
    print(f"{p['group_i']} vs {p['group_j']:<3} {p['meandiff']:8.3f} {p['lower']:8.3f} "
          f"{p['upper']:8.3f} {str(p['zero_in_interval']):>10} {str(p['reject']):>8} {p['p_adj']:8.4f}")

print()
print("statsmodels pairwise_tukeyhsd (library check)")
labels = np.array(["G1"] * 20 + ["G2"] * 20 + ["G3"] * 20)
print(pairwise_tukeyhsd(endog=np.concatenate([g1, g2, g3]), groups=labels, alpha=0.05))

visualization_utils.plot_fwer_and_tukey(tukey)
print("Red intervals: 0 is outside → reject that pair. Blue: 0 is inside → do not reject.")


### ANOVA simulation: three controlled cases

* **Case A** — all groups share a mean. The histogram of $F$ sits near $1$.
* **Case B** — means are separated, same noise. $F$ becomes large.
* **Case C** — same separation as B, but much more within-group noise. $F$ drops and power falls.

This is the power laboratory, rewritten in the language of $MS_B/MS_W$.


In [ ]:
visualization_utils.plot_anova_simulation_cases(n_reps=500, n=20, seed=5)


## 16. Unified Statistical Inference Experiment

One last pass through the whole chain, on a skewed population (and then on house prices).

**population → sample → estimator → sampling distribution → SE → CI → hypothesis test → effect size**


In [ ]:
# Synthetic skewed world
rng = np.random.default_rng(42)
population = rng.exponential(scale=5.0, size=100_000)
sample = rng.choice(population, size=50, replace=True)
summary = visualization_utils.plot_unified_inference_chain(population, sample, mu0=4.0, seed=42)

mu = summary["mu"]
est, se = summary["estimate"], summary["se"]
lo, hi = summary["ci"]
print(f"True μ (unknown in a real problem) = {mu:.3f}")
print(f"Estimate X̄ = {est:.3f},  SE = {se:.3f}")
print(f"95% t-interval = [{lo:.3f}, {hi:.3f}]")
print(f"H0: μ=4.0   t={summary['t']:.3f}, p={summary['p']:.4f}")
print(f"Cohen's d vs H0 = {(est - 4.0) / (se * np.sqrt(len(sample))):.3f}")
print("The population is exponential. The sampling distribution of X̄ is already close to normal.")


In [ ]:
# The same chain on the California housing population.
housing_sample = np.random.default_rng(7).choice(housing, size=40, replace=True)
h_sum = visualization_utils.plot_unified_inference_chain(housing, housing_sample, mu0=2.0, seed=7)
print(f"Housing mu = {h_sum['mu']:.3f},  Xbar = {h_sum['estimate']:.3f},  "
      f"CI=[{h_sum['ci'][0]:.3f}, {h_sum['ci'][1]:.3f}],  t={h_sum['t']:.3f}, p={h_sum['p']:.4f}")


## 17. Final Synthesis

We use finite samples to reason about unknown populations while explicitly accounting for uncertainty.

1. **Sample statistics are random variables** because samples are random.
2. **$Var(\bar X)=\sigma^2/n$** follows from $Var(aX)=a^2Var(X)$ and independence.
3. **$SE(\bar X)=\sigma/\sqrt n$** is the standard deviation of that sampling distribution, not of the raw data.
4. **Uncertainty falls slowly:** an $x$-fold tighter SE costs about $x^2$ observations.
5. **The CLT** says the *sampling distribution of the mean* becomes approximately $N(\mu,\sigma/\sqrt n)$. The population itself need not be normal.
6. An **estimator** is a rule; an **estimate** is a number.
7. **Bias** is $E[\hat\theta]-\theta$. The sample mean is unbiased for $\mu$.
8. **Variance of an estimator** is how much the rule jitters across samples. $X_1$ is unbiased and high-variance.
9. **$MSE=Var+Bias^2$**, derived by adding and subtracting $E[\hat\theta]$ and watching the cross term vanish.
10. **Consistency** is concentration as $n\to\infty$ (LLN). It is not the same as unbiasedness.
11. A **confidence interval** quantifies uncertainty around one estimate. The procedure has coverage; a finished interval does not carry a posterior probability. The half-width uses the **inverse CDF**: $t^*=F_{t_{n-1}}^{-1}(1-\alpha/2)$.
12. The **$t$ distribution** appears because $s$ replaces $\sigma$. Heavier tails, $df=n-1$, Normal in the limit.
13. A **$p$-value** is $P(\text{data this extreme or more}\mid H_0)$. It is not $P(H_0\mid\text{data})$.
14. The conversion **statistic $\to p$** is a tail probability of the null distribution: $p=1-F(t)$ (right tail), $p=F(t)$ (left tail), $p=2(1-F(|t|))$ (two-sided). Software evaluates a CDF / survival function (continuous) or sums a PMF (discrete).
15. You cannot get $p$ from $t$ alone: you also need the null law and the alternative's direction.
16. The **critical region** at $\alpha=0.05$ for a right-tailed $z$-test is $Z\geq Z_c=\Phi^{-1}(0.95)\approx 1.645$. Two-sided uses $\pm 1.96$. $p<\alpha$ iff the statistic lands in that region.
17. **Type I** is a false rejection ($\alpha$, the null mass in $C$). **Type II** is a missed effect ($\beta$, the alternative mass outside $C$).
18. **Power** is $1-\beta$. For a $z$-test, $\mathrm{Power}=\Phi(d\sqrt n-Z_c)$ with Cohen's $d=\delta/\sigma$. Larger effect, larger $n$, or smaller noise (variation) all make $H_0$ and $H_1$ easier to tell apart.
19. **One-sample** tests a mean against a number; **Welch** uses $SE=\sqrt{s_A^2/n_A+s_B^2/n_B}$ and Satterthwaite $df$; **pooled** assumes equal variance; **paired** tests differences and cancels shared variation.
20. **McNemar** is the paired test for binary classifier outcomes: $\chi^2=(b-c)^2/(b+c)$ referred to $\chi^2_1$, or an exact $\mathrm{Binomial}(b+c,1/2)$ tail sum. $Z^2\sim\chi^2_1$ makes the two-sided normal $p$-value match.
21. A $95\%$ CI and a two-sided test at $\alpha=0.05$ reject exactly when the null value sits outside the interval.
22. **Statistical significance $\neq$ practical significance.** Huge $n$ makes tiny effects "significant."
23. **ANOVA** compares between-group variation to within-group variation for $k\geq 3$ means. $p=P(F_{k-1,N-k}\geq F_{\mathrm{obs}})$.
24. **$SS_T=SS_B+SS_W$** because the cross term $\sum(X_{ij}-\bar X_j)(\bar X_j-\bar X)$ is zero.
25. **$F=MS_B/MS_W$**. Under $H_0$, $F\approx 1$; separated means drive $F\gg 1$.
26. Uncorrected pairwise tests inflate **FWER** $=1-(1-\alpha)^m$. A significant ANOVA means "at least one mean differs."
27. **Tukey HSD** compares pairs with the studentized range $q_{ij}$. The simultaneous CI is $\bar X_i-\bar X_j\pm q^*\sqrt{MS_W(1/n_i+1/n_j)/2}$. **If $0$ is outside the interval, reject that pair; if $0$ is inside, do not.**

The single picture:
$$
\boxed{\text{How we use finite samples to reason about unknown populations, without pretending the sample is the truth.}}
$$
